In [4]:
%pip install --upgrade --force-reinstall numpy==1.26.4 pandas==2.2.3 scipy==1.13.1 scikit-learn==1.5.2 pyarrow==17.0.0 pgmpy==1.0.0

  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
     ---------------------------------------- 0.0/60.6 kB ? eta -:--:--
     ---------------------------------------- 0.0/60.6 kB ? eta -:--:--
     ---------------------------------------- 0.0/60.6 kB ? eta -:--:--
     ---------------------------------------- 0.0/60.6 kB ? eta -:--:--
     -------------------- ------------------- 30.7/60.6 kB ? eta -:--:--
     -------------------- ------------------- 30.7/60.6 kB ? eta -:--:--
     -------------------------------------- 60.6/60.6 kB 535.5 kB/s eta 0:00:00
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached statsmodels-0.15.0-cp312-cp312-win_amd64.whl.metadata (9.6 kB)
     ---------------------------------------- 0.0/57.4 kB ? eta -:--:--
     ----------------------------------- ---- 51.2/57.4 kB 2.6 MB/s eta 0:00:01
     ---------------------------------------- 57.4/57.4 kB 1.5 MB/s eta 0:00:00
  Using cached opt_einsum-3.4.0-py3-none-a

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.12.3 requires botocore<1.34.70,>=1.34.41, but you have botocore 1.40.50 which is incompatible.
aiobotocore 2.12.3 requires wrapt<2.0.0,>=1.10.10, but you have wrapt 2.5.2 which is incompatible.
astroid 2.14.2 requires wrapt<2,>=1.14; python_version >= "3.11", but you have wrapt 2.5.2 which is incompatible.
botocore 1.40.50 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 4.0.0 which is incompatible.
category-encoders 2.8.1 requires scikit-learn>=1.6.0, but you have scikit-learn 1.5.2 which is incompatible.
graphene 3.4.3 requires python-dateutil<3,>=2.7.0, but you 

In [2]:
import numpy as np
import pandas as pd
import scipy
import sklearn
import pyarrow
import pgmpy


NumPy: 1.26.4
Pandas: 2.2.3
SciPy: 1.13.1
Scikit-learn: 1.5.2
PyArrow: 17.0.0
pgmpy: 1.0.0
All libraries imported successfully!


In [15]:
# Import required libraries
import numpy as np
import pandas as pd

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

# Read Heart Disease dataset
heartDisease = pd.read_csv('P3 heart.csv')

# Replace '?' with NaN
heartDisease = heartDisease.replace('?', np.nan)

# Rename target column
heartDisease = heartDisease.rename(
    columns={'target': 'heartdisease'}
)

# Select required columns
required_columns = [
    'age', 'sex', 'exang', 'trestbps',
    'fbs', 'heartdisease', 'restecg',
    'thalach', 'chol'
]

# Convert values to numeric
for col in required_columns:
    heartDisease[col] = pd.to_numeric(
        heartDisease[col], errors='coerce'
    )

# Remove missing values
heartDisease = heartDisease.dropna(
    subset=required_columns
)

# Display first five rows
print("Few examples from the dataset are given below:\n")
print(heartDisease.head())

# Define Bayesian Network
model = DiscreteBayesianNetwork([
    ('age', 'trestbps'),
    ('age', 'fbs'),
    ('sex', 'trestbps'),
    ('exang', 'trestbps'),
    ('trestbps', 'heartdisease'),
    ('fbs', 'heartdisease'),
    ('heartdisease', 'restecg'),
    ('heartdisease', 'thalach'),
    ('heartdisease', 'chol')
])

# Learn CPDs using Maximum Likelihood Estimation
print("\nLearning CPD using Maximum Likelihood Estimators...")

model.fit(
    heartDisease[required_columns],
    estimator=MaximumLikelihoodEstimator
)

print("Model Learned Successfully!")

# Check model validity
print("Model Valid:", model.check_model())

# Perform inference
print("\nInferencing with Bayesian Network:")

HeartDisease_infer = VariableElimination(model)

# Probability given Age = 38
print("\n1. Probability of Heart Disease given Age = 38")

q = HeartDisease_infer.query(
    variables=['heartdisease'],
    evidence={'age': 38}
)

print(q)

# Probability given Cholesterol = 230
print("\n2. Probability of Heart Disease given Cholesterol = 230")

q = HeartDisease_infer.query(
    variables=['heartdisease'],
    evidence={'chol': 230}
)

print(q)

print("\nPractical Executed Successfully!")

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'age': 'N', 'sex': 'N', 'exang': 'N', 'trestbps': 'N', 'fbs': 'N', 'heartdisease': 'N', 'restecg': 'N', 'thalach': 'N', 'chol': 'N'}


Few examples from the dataset are given below:

   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   52    1   0       125   212    0        1      168      0      1.0      2   
1   53    1   0       140   203    1        0      155      1      3.1      0   
2   70    1   0       145   174    0        1      125      1      2.6      0   
3   61    1   0       148   203    0        1      161      0      0.0      2   
4   62    0   0       138   294    1        1      106      0      1.9      1   

   ca  thal  heartdisease  
0   2     3             0  
1   0     3             0  
2   0     3             0  
3   1     3             0  
4   3     2             0  

Learning CPD using Maximum Likelihood Estimators...
Model Learned Successfully!
Model Valid: True

Inferencing with Bayesian Network:

1. Probability of Heart Disease given Age = 38
+-----------------+---------------------+
| heartdisease    |   phi(heartdisease) |
+=================+===========

# Practical 3: Bayesian Network for Heart Disease Diagnosis
## Oral / Viva Questions and Answers

### 1. What is a Bayesian Network?
A Bayesian Network is a probabilistic graphical model that represents relationships between variables using a directed acyclic graph (DAG).

### 2. What is the main objective of this practical?
To construct a Bayesian Network using heart disease medical data and calculate the probability of heart disease using inference.

### 3. What is a Bayesian Network used for?
It is used for prediction, diagnosis, decision-making, and reasoning under uncertainty.

### 4. What is a DAG?
DAG stands for Directed Acyclic Graph. It is a graph with directed edges and no directed cycles.

### 5. What are nodes in a Bayesian Network?
Nodes represent random variables such as age, cholesterol, blood pressure, and heart disease.

### 6. What are edges in a Bayesian Network?
Edges represent dependency or probabilistic relationships between variables.

### 7. What is a Conditional Probability Distribution (CPD)?
A CPD describes the probability of a variable based on its parent variables.

### 8. What is Maximum Likelihood Estimation?
Maximum Likelihood Estimation (MLE) is a method used to estimate model probabilities from observed data.

### 9. Which library is used to construct the Bayesian Network?
We use the pgmpy Python library.

### 10. What is pgmpy?
pgmpy is a Python library used for probabilistic graphical models, including Bayesian Networks.

### 11. Which dataset is used in this practical?
The Heart Disease Dataset, containing medical information about patients.

### 12. What is the target variable?
The target variable represents whether the patient has heart disease.

### 13. What does target = 0 represent?
It represents the absence of heart disease in this dataset.

### 14. What does target = 1 represent?
It represents the presence of heart disease in this dataset.

### 15. What is the purpose of NumPy?
NumPy is used for numerical operations and handling missing values using NaN.

### 16. What is the purpose of Pandas?
Pandas is used to load, clean, and manipulate the dataset.

### 17. Why do we replace '?' with NaN?
Because '?' represents missing data, and NaN allows us to identify and handle missing values.

### 18. What is the purpose of dropna()?
It removes rows containing missing values in the selected columns.

### 19. What is DiscreteBayesianNetwork?
It is a pgmpy class used to create Bayesian Networks containing discrete variables.

### 20. Why do we use MaximumLikelihoodEstimator?
It learns the Conditional Probability Distributions from the dataset.

### 21. What is model.fit()?
It learns the CPDs of the Bayesian Network using the given dataset and estimator.

### 22. What is Variable Elimination?
Variable Elimination is an inference algorithm used to calculate probabilities in a Bayesian Network.

### 23. Why do we use VariableElimination?
To calculate the probability of heart disease based on given evidence.

### 24. What is evidence in a Bayesian Network?
Evidence is known information about a variable that is used to calculate conditional probabilities.

### 25. What is inference?
Inference is the process of calculating unknown probabilities using known evidence.

### 26. What does evidence={'age': 38} mean?
It means we calculate the probability distribution of heart disease when the patient's age is 38.

### 27. What does evidence={'chol': 230} mean?
It means we calculate the probability distribution of heart disease when the patient's cholesterol level is 230.

### 28. What is cholesterol?
Cholesterol is a fatty substance present in the blood. Its level is one of the medical features in the dataset.

### 29. What is trestbps?
It represents resting blood pressure.

### 30. What is thalach?
It represents the maximum heart rate achieved by the patient.

### 31. What is fbs?
FBS stands for fasting blood sugar.

### 32. What is exang?
Exang represents exercise-induced angina.

### 33. What is restecg?
It represents resting electrocardiographic results.

### 34. What is the purpose of model.check_model()?
It checks whether the Bayesian Network structure and its CPDs are valid.

### 35. What is the difference between prediction and inference?
Prediction estimates an unknown outcome, while inference calculates probabilities using available evidence.

### 36. What is the advantage of a Bayesian Network?
It handles uncertainty and represents probabilistic relationships between variables.

### 37. What is the limitation of a Bayesian Network?
Its accuracy depends on the quality of data and the correctness of the network structure.

### 38. Why is heart disease diagnosis suitable for a Bayesian Network?
Because medical diagnosis involves uncertainty and relationships between multiple medical factors.

### 39. Can a Bayesian Network handle missing data?
Yes, Bayesian Networks can perform inference with incomplete evidence, although missing training data must be handled appropriately.

### 40. What is the final outcome of this practical?
We successfully construct a Bayesian Network, learn probabilities from heart disease data, and perform inference to estimate heart disease probabilities.

---

## Important Code-Based Viva Questions

### 41. Why do we use BayesianNetwork or DiscreteBayesianNetwork?
To define the network structure. Our installed pgmpy version uses DiscreteBayesianNetwork.

### 42. What does model.fit() return?
It fits the model with learned CPDs and updates the model.

### 43. What does query() do?
It calculates the probability distribution of the requested variable given the evidence.

### 44. Why do we use variables=['heartdisease']?
Because we want to calculate the probability distribution of the heart disease variable.

### 45. What is the output of the practical?
The output is a probability distribution showing the likelihood of heart disease outcomes for the given evidence.